<a href="https://colab.research.google.com/github/kamalrawat77/agentic-iam-lab/blob/main/week06-rag-foundations/Nugget030_First_RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nugget 026: Persistent Investigation Memory

In [36]:
!pip install -q google-genai
!pip install sentence-transformers
import json
from sentence_transformers import SentenceTransformer
from sentence_transformers import util

In [37]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [38]:
from google import genai

def callGPT(prompt):
  client = genai.Client(api_key="APIKEY")
  response = client.models.generate_content(
      model="gemini-2.5-flash",
      contents=prompt
  )

  return response

In [39]:
def cleanse_response(response):
  clean_response=clean_response = response.text
  clean_response = clean_response.replace("```json", "")
  clean_response = clean_response.replace("```", "")
  clean_response = clean_response.strip()

  return clean_response

In [40]:
def return_json(clean_response,objname):
  responseObj = json.loads(clean_response)
  resJsonObj  = responseObj[objname]
  return resJsonObj



In [41]:
def save_investigation(memory, investigation):

    memory.append(investigation)

    return memory

In [42]:
def get_last_investigation(memory):

    if len(memory) == 0:
        return None

    return memory[-1]

Load investigation history

Copy data from github project data/investigation_memory.json

In [43]:
data=[
  {
    "investigation_id": 1,
    "date": "2026-06-19",
    "dormant_accounts": 70,
    "inactive_approvers": 35,
    "sla_compliance": 72,
    "root_cause": "Inactive Approvers",
    "confidence": "HIGH",
    "recommendation": "Enable approver monitoring"
  },
  {
    "investigation_id": 2,
    "date": "2026-06-20",
    "dormant_accounts": 75,
    "inactive_approvers": 25,
    "sla_compliance": 70,
    "root_cause": "Workflow Bottleneck",
    "confidence": "HIGH",
    "recommendation": "Optimize workflow routing"
  },
  {
    "investigation_id": 3,
    "date": "2026-06-21",
    "dormant_accounts": 65,
    "inactive_approvers": 25,
    "sla_compliance": 35,
    "root_cause": "Provisioning Delays",
    "confidence": "HIGH",
    "recommendation": "Increase connector capacity"
  },
  {
    "investigation_id": 4,
    "date": "2026-06-22",
    "dormant_accounts": 85,
    "inactive_approvers": 15,
    "sla_compliance": 72,
    "root_cause": "Inactive Approvers",
    "confidence": "HIGH",
    "recommendation": "Enable approver monitoring"
  },
  {
    "investigation_id": 5,
    "date": "2026-06-23",
    "dormant_accounts": 115,
    "inactive_approvers": 115,
    "sla_compliance": 72,
    "root_cause": "Inactive Approvers",
    "confidence": "HIGH",
    "recommendation": "Enable approver monitoring"
  }
]

Load data to current notebook context

In [44]:
with open("investigation_memory.json", "w") as f:
    json.dump(data, f)

Load investigation history now

In [45]:
with open("investigation_memory.json") as f:
    investigations = json.load(f)

print(investigations)

[{'investigation_id': 1, 'date': '2026-06-19', 'dormant_accounts': 70, 'inactive_approvers': 35, 'sla_compliance': 72, 'root_cause': 'Inactive Approvers', 'confidence': 'HIGH', 'recommendation': 'Enable approver monitoring'}, {'investigation_id': 2, 'date': '2026-06-20', 'dormant_accounts': 75, 'inactive_approvers': 25, 'sla_compliance': 70, 'root_cause': 'Workflow Bottleneck', 'confidence': 'HIGH', 'recommendation': 'Optimize workflow routing'}, {'investigation_id': 3, 'date': '2026-06-21', 'dormant_accounts': 65, 'inactive_approvers': 25, 'sla_compliance': 35, 'root_cause': 'Provisioning Delays', 'confidence': 'HIGH', 'recommendation': 'Increase connector capacity'}, {'investigation_id': 4, 'date': '2026-06-22', 'dormant_accounts': 85, 'inactive_approvers': 15, 'sla_compliance': 72, 'root_cause': 'Inactive Approvers', 'confidence': 'HIGH', 'recommendation': 'Enable approver monitoring'}, {'investigation_id': 5, 'date': '2026-06-23', 'dormant_accounts': 115, 'inactive_approvers': 115,

Create searchable text.

In [46]:
documents = []

for inv in investigations:

    documents.append(
        f"""
        Root Cause:
        {inv['root_cause']}

        Recommendation:
        {inv['recommendation']}
        """
    )

Generate Embeddings

In [47]:
embeddings = model.encode(documents)

Create retrieval function and test it

In [55]:
def retrieve(query):

    query_embedding = model.encode(query)

    scores = util.cos_sim(
        query_embedding,
        embeddings
    )

    best_index = scores.argmax()

    return documents[best_index]

In [79]:
def retrieve_top3(query):

    query_embedding = model.encode(query)

    scores = util.cos_sim(
        query_embedding,
        embeddings
    )

    top_indices = scores[0].argsort(descending=True)[:4]

    contexts = []

    for idx in top_indices:
        #print(documents[idx])
        contexts.append(documents[idx])

    context = "\n\n".join(contexts)
    return context

In [80]:
context = retrieve_top3(
    "Managers are not responding"
)

print(context)


        Root Cause:
        Inactive Approvers

        Recommendation:
        Enable approver monitoring
        


        Root Cause:
        Inactive Approvers

        Recommendation:
        Enable approver monitoring
        


        Root Cause:
        Inactive Approvers

        Recommendation:
        Enable approver monitoring
        


        Root Cause:
        Provisioning Delays

        Recommendation:
        Increase connector capacity
        


Build a RAG now

In [87]:
question = """
Managers are not responding
and approvals are delayed.
"""

In [88]:
context = retrieve_top3(question)

Prompt LLM

In [89]:
prompt = f"""
You are an IAM analyst.

Question:

{question}

Relevant Historical Context:

{context}

Answer the question using the context.
"""

In [90]:
print(callGPT(prompt).text)

Based on the historical context provided:

The root cause of "Managers are not responding and approvals are delayed" is most likely **Inactive Approvers**.

The recommended action to address this issue is to **Enable approver monitoring**.
